# Project: BBLF AI Selector v2 
# Section: Model Scoring - Expected Points Model
## Sub Section: During Tourny (Post Rnd 6)

Goal: Develop streamline scoring process to predict the players expected points for BBL 15

In [5]:
# pip install --upgrade scikit-learn xgboost interpret

# Prerequistes

In [6]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import joblib

import warnings
warnings.filterwarnings("ignore")

os.getcwd()
model_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/python_script/model/build/models/'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/post_round_6/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_data_score_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/scoring/post-round-6/'

# Import Model Object

In [7]:
# 1. Load model object
model_obj_pt = joblib.load(os.path.join(model_directory,'ps_all_mdl_1'))
model_obj_pg1 = joblib.load(os.path.join(model_directory,'ps_all_mdl_pg1_3'))
model_obj_pg2 = joblib.load(os.path.join(model_directory,'ps_all_mdl_pg2_4'))
model_obj_pg3 = joblib.load(os.path.join(model_directory,'ps_all_mdl_pg3_4'))
model_obj_pg4 = joblib.load(os.path.join(model_directory,'ps_all_mdl_pg4_3'))
model_obj_pg5 = joblib.load(os.path.join(model_directory,'ps_all_mdl_pg5_3'))

# 2. Model Feature Names
feat_list_pt = model_obj_pt.get_booster().feature_names
feat_list_pg1 = model_obj_pg1.get_booster().feature_names
feat_list_pg2 = model_obj_pg2.get_booster().feature_names
feat_list_pg3 = model_obj_pg3.get_booster().feature_names
feat_list_pg4 = model_obj_pg4.get_booster().feature_names
feat_list_pg5 = model_obj_pg5.get_booster().feature_names


# Import Datasets required for Model Scoring

In [8]:
# 1. Import Datasets
# All BBL 15 Player & their Team Data
player_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026_efp_rnd_6.csv'), low_memory=False)
# player_df = player_df[["Full_Name","player", "Team"]].rename(columns = {"Full_Name":"Name"})
player_df = player_df[["Full_Name", "player", "Team"]]

# BBL 15 Player Features
# Prior Season Features (Lags)
lags_15_df = pd.read_csv(os.path.join(py_data_score_directory,'bbl15_lags_pr6.csv'), low_memory=False)
lags_15_df = lags_15_df.drop(["player"], axis = 1)
# lags_15_df = lags_15_df.drop(["Unnamed: 0"], axis = 1)

# BBL14 Fixture
# Need a table for each team with opposition and where they play against the opposition
team_venue_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
team_venue_df['Game'] = team_venue_df.groupby('Team').cumcount() + 1

# 2. Join Datasets
# Join BBL15 Player Team Data - All Player Fixture Possible Scenarios
player_fix_scen_df = pd.merge(player_df , team_venue_df, left_on = ["Team"], right_on = ["Team"], how = "left")
player_fix_scen_df = player_fix_scen_df.rename(columns = {"Opposition":"opp", "Venue":"venue"})

## Join BBL14 Player Feature Data - All Player Fixture Possible Scenarios
bbl15_scen_df = pd.merge(player_fix_scen_df , lags_15_df, left_on = ["Full_Name"], right_on = ["Full_Name"], how = "left").rename(columns = {"Full_Name":"Name"})
print(bbl15_scen_df)


               Name         player                 Team                opp  \
0     Matthew Short       MW Short    Adelaide Strikers      Sydney Sixers   
1     Matthew Short       MW Short    Adelaide Strikers    Melbourne Stars   
2     Matthew Short       MW Short    Adelaide Strikers      Brisbane Heat   
3     Matthew Short       MW Short    Adelaide Strikers      Brisbane Heat   
4     Matthew Short       MW Short    Adelaide Strikers    Perth Scorchers   
...             ...            ...                  ...                ...   
1415  Will Salzmann  Will Salzmann  Melbourne Renegades    Perth Scorchers   
1416  Will Salzmann  Will Salzmann  Melbourne Renegades    Melbourne Stars   
1417  Will Salzmann  Will Salzmann  Melbourne Renegades     Sydney Thunder   
1418  Will Salzmann  Will Salzmann  Melbourne Renegades    Perth Scorchers   
1419  Will Salzmann  Will Salzmann  Melbourne Renegades  Adelaide Strikers   

                  venue  Home_f  Round  Game  season_fp_lag1  \

# Override Predictions with Actual Values

In [9]:
# 1. Extract Actual Points Data
bbl15_prior_df = pd.read_csv(os.path.join(add_data_directory,'player_bbl15_pts_rnd_6.csv'), low_memory=False)
bbl15_prior_df = bbl15_prior_df.drop(['Team','Price'], axis = 1)
bbl15_prior_df.columns = bbl15_prior_df.columns.str.replace(r'Game_', '')
bbl15_prior_df = pd.melt(bbl15_prior_df, id_vars=['player'], value_vars= ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10'],
                         var_name='Game', value_name='fantasy_point').dropna()
bbl15_prior_df['Game'] = bbl15_prior_df['Game'].astype(int)
bbl15_prior_df = bbl15_prior_df.rename(columns={"player": "Name"})

# 2. Override Expected Points with Actual Points where available
bbl15_scen_df = pd.merge(bbl15_scen_df, bbl15_prior_df, left_on = ["Name","Game"], right_on = ["Name","Game"], how = "left")

display(bbl15_scen_df.head(10))

,Name,player,Team,opp,venue,Home_f,Round,Game,season_fp_lag1,avg_season_fp_lag1,...,avg_bat_pos_8_curr,avg_bat_pos_9_curr,avg_bat_pos_10_curr,avg_bat_pos_11_curr,avg_bat_cat_Open_curr,avg_bat_cat_TM_curr,avg_bat_cat_LM_curr,avg_bat_cat_Tail_curr,avg_season_bat_pos_curr,fantasy_point
0,Matthew Short,MW Short,Adelaide Strikers,Sydney Sixers,SCG,0,1,1,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,46.0
1,Matthew Short,MW Short,Adelaide Strikers,Melbourne Stars,Adelaide Oval,1,2,2,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,95.0
2,Matthew Short,MW Short,Adelaide Strikers,Brisbane Heat,GABBA,0,3,3,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,175.0
3,Matthew Short,MW Short,Adelaide Strikers,Brisbane Heat,Adelaide Oval,1,4,4,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,42.0
4,Matthew Short,MW Short,Adelaide Strikers,Perth Scorchers,Perth Stadium,0,5,5,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,77.0
5,Matthew Short,MW Short,Adelaide Strikers,Sydney Thunder,Adelaide Oval,1,6,6,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,NaN
6,Matthew Short,MW Short,Adelaide Strikers,Hobart Hurricanes,Hobart,0,7,7,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,NaN
7,Matthew Short,MW Short,Adelaide Strikers,Perth Scorchers,Adelaide Oval,1,7,8,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,NaN
8,Matthew Short,MW Short,Adelaide Strikers,Melbourne Stars,Melbourne Cricket Ground,0,8,9,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,NaN
9,Matthew Short,MW Short,Adelaide Strikers,Melbourne Renegades,Adelaide Oval,1,9,10,508.0,72.571429,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,NaN


# Score BBL 15 Data using Model Object

In [10]:
# 1. Score Data

# a. Split model df based on current matches played
bbl15_play_scen_df_pt = bbl15_scen_df[(bbl15_scen_df['match_cnt_curr'].isna())]
bbl15_play_scen_df_pg1 = bbl15_scen_df[(bbl15_scen_df['match_cnt_curr'] == 1)]
bbl15_play_scen_df_pg2 = bbl15_scen_df[(bbl15_scen_df['match_cnt_curr'] == 2)]
bbl15_play_scen_df_pg3 = bbl15_scen_df[(bbl15_scen_df['match_cnt_curr'] == 3)]
bbl15_play_scen_df_pg4 = bbl15_scen_df[(bbl15_scen_df['match_cnt_curr'] == 4)]
bbl15_play_scen_df_pg5 = bbl15_scen_df[(bbl15_scen_df['match_cnt_curr'] >= 5)]

# b. Create model scoring df by matching train model df columns
bbl15_play_scen_df_model_pt = bbl15_play_scen_df_pt[feat_list_pt]
bbl15_play_scen_df_model_pg1 = bbl15_play_scen_df_pg1[feat_list_pg1]
bbl15_play_scen_df_model_pg2 = bbl15_play_scen_df_pg2[feat_list_pg2]
bbl15_play_scen_df_model_pg3 = bbl15_play_scen_df_pg3[feat_list_pg3]
bbl15_play_scen_df_model_pg4 = bbl15_play_scen_df_pg4[feat_list_pg4]
bbl15_play_scen_df_model_pg5 = bbl15_play_scen_df_pg5[feat_list_pg5]

# c. Player Expected Fantasy Point Scoring
bbl15_play_exp_fp_pt = model_obj_pt.predict(bbl15_play_scen_df_model_pt)
bbl15_play_exp_fp_pg1 = model_obj_pg1.predict(bbl15_play_scen_df_model_pg1)
bbl15_play_exp_fp_pg2 = model_obj_pg2.predict(bbl15_play_scen_df_model_pg2)
bbl15_play_exp_fp_pg3 = model_obj_pg3.predict(bbl15_play_scen_df_model_pg3)
bbl15_play_exp_fp_pg4 = model_obj_pg4.predict(bbl15_play_scen_df_model_pg4)
bbl15_play_exp_fp_pg5 = model_obj_pg5.predict(bbl15_play_scen_df_model_pg5)

# d. Join predictions to scoring dataframe
bbl15_scen_model_score_full_pt = bbl15_play_scen_df_pt.copy()
bbl15_scen_model_score_full_pg1 = bbl15_play_scen_df_pg1.copy()
bbl15_scen_model_score_full_pg2 = bbl15_play_scen_df_pg2.copy()
bbl15_scen_model_score_full_pg3 = bbl15_play_scen_df_pg3.copy()
bbl15_scen_model_score_full_pg4 = bbl15_play_scen_df_pg4.copy()
bbl15_scen_model_score_full_pg5 = bbl15_play_scen_df_pg5.copy()

bbl15_scen_model_score_full_pt["mdl_exp_pts"] = bbl15_play_exp_fp_pt
bbl15_scen_model_score_full_pg1["mdl_exp_pts"] = bbl15_play_exp_fp_pg1
bbl15_scen_model_score_full_pg2["mdl_exp_pts"] = bbl15_play_exp_fp_pg2
bbl15_scen_model_score_full_pg3["mdl_exp_pts"] = bbl15_play_exp_fp_pg3
bbl15_scen_model_score_full_pg4["mdl_exp_pts"] = bbl15_play_exp_fp_pg4
bbl15_scen_model_score_full_pg5["mdl_exp_pts"] = bbl15_play_exp_fp_pg5

# # e. Combine all model scoring dfs by columns
bbl15_scen_model_score_full = pd.concat([bbl15_scen_model_score_full_pt,
                                   bbl15_scen_model_score_full_pg1,
                                   bbl15_scen_model_score_full_pg2,
                                   bbl15_scen_model_score_full_pg3,
                                   bbl15_scen_model_score_full_pg4,
                                   bbl15_scen_model_score_full_pg5,
                                   ], ignore_index=True)

# f. Where actual points are available, override expected points with actual points
bbl15_scen_model_score_full["exp_pts"] = np.where(bbl15_scen_model_score_full["fantasy_point"].isna(),
                                                    bbl15_scen_model_score_full["mdl_exp_pts"],
                                                    bbl15_scen_model_score_full["fantasy_point"])
# g. Flag to indicate if points are actual or expected
bbl15_scen_model_score_full["pts_type"] = np.where(bbl15_scen_model_score_full["fantasy_point"].isna(),
                                                    "exp",
                                                    "actl")

# Short Version - Only Key Columns 
bbl15_scen_model_score_short = bbl15_scen_model_score_full[["player","Name","Team","Round","opp","venue","Home_f","exp_pts","pts_type"]]


# Save Model Scoring Dataframes

In [11]:
# Save Model Scoring DataFrame
bbl15_scen_model_score_full.to_csv(os.path.join(py_data_score_directory,'bbl15_efp_model_score_full_pr6.csv'), index=False)
bbl15_scen_model_score_short.to_csv(os.path.join(py_data_score_directory,'bbl15_efp_model_score_short_pr6.csv'), index=False)